# Q4: Lattice-Based ASR Evaluation

Constructs lattices from **5 ASR model outputs + human reference** and computes fairer WER that accounts for valid transcription alternatives.

Uses the Question 4 CSV data (46 segments).

## Setup & Imports

In [ ]:
import os
import sys
import pandas as pd
import numpy as np

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, PROJECT_ROOT)

from src.lattice_builder import (
    build_lattice, lattice_to_string,
    compute_standard_wer, compute_lattice_wer,
    evaluate_with_lattice, tokenize
)

print(f"Project root: {PROJECT_ROOT}")

## STEP 1: Load & Explore Data

In [ ]:
print("=" * 60)
print("Q4: Lattice-Based ASR Evaluation")
print("=" * 60)

CSV_PATH = os.path.join(PROJECT_ROOT, "Question 4 - Task.csv")
df = pd.read_csv(CSV_PATH)

print(f"Total segments: {len(df)}")
print(f"Columns: {df.columns.tolist()}")

model_columns = [col.strip() for col in df.columns if col.strip() not in ['segment_url_link', 'Human', '']]
print(f"Models: {model_columns}")

## STEP 2: Alignment Unit Justification

### Alignment Unit Choice: **WORD-LEVEL**

**Justification:**

1. Hindi is predominantly space-delimited, making word-level the most natural alignment unit.
2. Subword chunking would fragment Hindi's agglutinative morphology and lose semantic meaning.
3. Phrase-level alignment is too coarse — it would mask individual word-level variations.
4. **Special handling:** Compound word variants (with/without space) are treated as valid alternatives via edit-distance-based alignment.
5. Punctuation is stripped before alignment since Hindi ASR typically doesn't predict punctuation consistently.

## STEP 3: Lattice Construction Demo

In [ ]:
print("=" * 60)
print("STEP 3: Lattice Construction Demo")
print("=" * 60)

# Use first few segments as demo
for idx in range(min(5, len(df))):
    row = df.iloc[idx]
    human = str(row['Human'])
    models = [str(row[col]) if pd.notna(row[col]) else "" for col in model_columns]
    
    lattice = build_lattice(human, models)
    
    print(f"\n--- Segment {idx+1} ---")
    print(f"  Human: {human[:70]}...")
    print(f"  Lattice: {lattice_to_string(lattice)[:100]}...")
    print(f"  Bins: {len(lattice)}, Bins with alternatives: {sum(1 for b in lattice if len(b) > 1)}")

## STEP 4: Compute Standard vs Lattice WER

In [ ]:
print("=" * 60)
print("STEP 4: Compute WER (Standard vs Lattice)")
print("=" * 60)

OUTPUT_PATH = os.path.join(PROJECT_ROOT, "results", "q4_lattice_wer_results.csv")
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

detailed_results, summary = evaluate_with_lattice(CSV_PATH, OUTPUT_PATH)

print("\n" + "=" * 60)
print("FINAL WER COMPARISON TABLE")
print("=" * 60)
print(f"\n{'Model':<12} {'Standard WER':>14} {'Lattice WER':>13} {'Reduction':>11} {'Improvement%':>14}")
print("-" * 65)
for model_name, row in summary.iterrows():
    print(f"{model_name:<12} {row['standard_wer']:>14.4f} {row['lattice_wer']:>13.4f} {row['wer_reduction']:>11.4f} {row['improvement_%']:>13.2f}%")

## STEP 5: Analysis — Who Benefits and Why

In [ ]:
print("=" * 60)
print("STEP 5: Analysis")
print("=" * 60)

# Find models that benefit most/least
best_improvement = summary['improvement_%'].idxmax()
worst_improvement = summary['improvement_%'].idxmin()

print(f"""
ANALYSIS:

1. MOST BENEFITED MODEL: {best_improvement}
   - Standard WER: {summary.loc[best_improvement, 'standard_wer']:.4f}
   - Lattice WER:  {summary.loc[best_improvement, 'lattice_wer']:.4f}
   - Improvement:  {summary.loc[best_improvement, 'improvement_%']:.2f}%
   - REASON: This model likely produces valid alternative transcriptions that
     differ from the human reference but are not actually wrong. The lattice
     captures these valid alternatives, removing unfair penalties.

2. LEAST BENEFITED MODEL: {worst_improvement}
   - Standard WER: {summary.loc[worst_improvement, 'standard_wer']:.4f}
   - Lattice WER:  {summary.loc[worst_improvement, 'lattice_wer']:.4f}
   - Improvement:  {summary.loc[worst_improvement, 'improvement_%']:.2f}%
   - REASON: This model's errors are genuine recognition mistakes, not valid
     alternatives. The lattice correctly keeps their WER similar.

3. TRUST MECHANISM:
   When 3+ models agree on a word but the human reference differs, the lattice
   includes BOTH the human reference and the model consensus as valid alternatives.
   This prevents penalizing models for human transcription errors.

4. KEY INSIGHT:
   Lattice-based evaluation reduces WER for models that were unfairly penalized
   (valid alternatives counted as errors) while keeping WER unchanged for models
   with genuine errors. This validates the approach.
""")

In [ ]:
# Detailed examples of lattice benefit
print("--- EXAMPLES WHERE LATTICE HELPS ---")
for idx in range(min(3, len(df))):
    row = df.iloc[idx]
    human = str(row['Human'])
    models = {col: str(row[col]) if pd.notna(row[col]) else "" for col in model_columns}
    
    lattice = build_lattice(human, list(models.values()))
    
    # Find bins with alternatives
    alt_bins = [(i, b) for i, b in enumerate(lattice) if len(b) > 1]
    if alt_bins:
        print(f"\n  Segment {idx+1}: {human[:50]}...")
        for bin_idx, bin_set in alt_bins[:3]:
            print(f"    Position {bin_idx}: {sorted(bin_set)}")

## STEP 6: Save Comprehensive Results

In [ ]:
print("=" * 60)
print("STEP 6: Final Deliverables")
print("=" * 60)

# Save summary
summary_path = os.path.join(PROJECT_ROOT, "results", "q4_wer_summary.csv")
summary.to_csv(summary_path)

print(f"""
  ✓ Detailed per-segment results: {OUTPUT_PATH}
  ✓ Summary table: {summary_path}
  ✓ Alignment unit: Word-level (justified above)
  ✓ Lattice construction: Progressive MSA + Needleman-Wunsch
  ✓ Trust mechanism: model consensus (≥3 agree) adds alternatives
""")

print("\n✓ Q4 Complete!")